In [60]:
import os
import json
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, random_split, DataLoader
import torchvision.models as models
from torchvision.transforms import v2 as transforms
import kagglehub

In [61]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [62]:
path = kagglehub.dataset_download("desirewaheed/3d-body-mesh")
data_dir= os.path.join(path, "test")
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToImage(),
    transforms.ToDtype(torch.float32, scale=True),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [63]:
class BodyMeshDataset(Dataset):
    def __init__(self, root, transform=None):
        self.images_dir = os.path.join(root, "images")
        self.labels_dir = os.path.join(root, "labels")
        self.transform = transform

        self.filenames = sorted([
            f for f in os.listdir(self.images_dir) 
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])
        self.classes = ["body_mesh"]

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        name = self.filenames[idx]
        img_path = os.path.join(self.images_dir, name)
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
            
        base_name = os.path.splitext(name)[0]
        label_path = os.path.join(self.labels_dir, base_name + ".json") 
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                data =json.load(f)
                label_data = data["pose"]
                target = torch.tensor(label_data, dtype=torch.float32)
        else:
            target = torch.zeros(72, dtype=torch.float32)
            
        return image, target

class PosePredictionModel(nn.Module):
    def __init__(self,  output_dim = 72):
        super(PosePredictionModel, self).__init__()
        self.backbone = models.resnet34(weights = models.ResNet34_Weights.DEFAULT)
        num_features = self.backbone.fc.in_features
        self.backbone.fc =nn.Sequential(
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, output_dim)
        )

    def forward(self,x):
        return self.backbone(x)

dataset = BodyMeshDataset(root=data_dir, transform=transform)

num_classes = len(dataset.classes)
print(f"Total images found: {len(dataset)}")
print(f"Number of target classes: {num_classes}")

total_size = len(dataset)
train_size = int(0.80 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"Train size: {len(train_dataset)} | Val size: {len(val_dataset)} | Test size: {len(test_dataset)}")

Total images found: 28530
Number of target classes: 1
Train size: 22824 | Val size: 4279 | Test size: 1427


In [64]:
batch_size =  32
train_loader = DataLoader(train_dataset,batch_size=batch_size,shuffle= True, num_workers=2)
val_loader = DataLoader(val_dataset,batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset,batch_size=batch_size, shuffle=False, num_workers=2)

print(
    f"Train: {len(train_dataset)} | "
    f"Validation: {len(val_dataset)} | "
    f"Test: {len(test_dataset)}"
)

Train: 22824 | Validation: 4279 | Test: 1427


In [65]:
model = PosePredictionModel(output_dim=72).to(device)
criterion=nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr = 1e-4)

print(model)

PosePredictionModel(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=Tr

In [66]:
import time
num_epochs = 30
train_losses = []
val_losses = []

print(f"Starting tracking verification over {num_epochs} epoch...")
for epoch in range(num_epochs):
    start_time = time.time()
    
    model.train()
    running_train_loss = 0.0

    for batch_idx, (images,  targets) in enumerate(train_loader):
        images= images.to(device)
        targets = targets.to(device)

        outputs = model(images)
        loss= criterion(outputs,  targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()* images.size(0)

    epoch_train_loss = running_train_loss/len(train_dataset)

    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for images, targets in val_loader:
            images= images.to(device)
            targets = targets.to(device)

            outputs = model(images)
            loss= criterion(outputs,  targets)

            running_val_loss += loss.item()*images.size(0)

    epoch_val_loss = running_val_loss/len(val_dataset)

    epoch_time = time.time() -start_time

    print(f"Epoch [{epoch+1:02d}/{num_epochs:02d}] | "
          f"Train Loss: {epoch_train_loss:.6f} | "
          f"Val Loss: {epoch_val_loss:.6f} | "
          f"Time: {epoch_time:.1f}s")

print("DONE!!!")

Starting tracking verification over 30 epoch...


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e3245bf1ee0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e3245bf1ee0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Epoch [01/30] | Train Loss: 0.020064 | Val Loss: 0.017271 | Time: 203.8s


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e3245bf1ee0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e3245bf1ee0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()Exception ignored in: 
<function _MultiProcessingDataLoaderIter.__del__ at 0x7e3245bf1ee0>  File "/usr/local/lib/python3.13/dist-packages/torch/utils/da

Epoch [02/30] | Train Loss: 0.016236 | Val Loss: 0.014652 | Time: 204.3s


...

Epoch [28/30] | Train Loss: 0.003792 | Val Loss: 0.010194 | Time: 203.1s


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e3245bf1ee0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e3245bf1ee0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Epoch [29/30] | Train Loss: 0.003661 | Val Loss: 0.010164 | Time: 202.6s


Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e3245bf1ee0>
<function _MultiProcessingDataLoaderIter.__del__ at 0x7e3245bf1ee0>Traceback (most recent call last):

  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
Traceback (most recent call last):
      File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
self._shutdown_workers()
      File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
self._shutdown_workers()    
if w.is_alive():  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers

  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
        if w.is_alive():assert self._parent_pid == os.getpid(), 'can only test a child process'

AssertionError  File "/usr/lib/python3.13/multiprocessing/proce

Epoch [30/30] | Train Loss: 0.003617 | Val Loss: 0.010233 | Time: 202.9s
DONE!!!


In [68]:
import matplotlib.pyplot as plt

# Automatically find how many epochs actually completed training
completed_epochs = len(train_losses)

if completed_epochs == 0:
    print("Error: You haven't run the training loop yet! Please run your training loop cell first so there is data to plot.")
else:
    plt.figure(figsize=(10, 6))

    # Use completed_epochs instead of num_epochs
    plt.plot(range(1, completed_epochs + 1), train_losses, label='Training Loss', color='blue', linewidth=2)
    plt.plot(range(1, completed_epochs + 1), val_losses, label='Validation Loss', color='orange', linewidth=2)

    plt.title('3D Body Mesh Model - Pose Regression Loss Over Epochs', fontsize=14, fontweight='bold')
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Loss (Mean Squared Error)', fontsize=12)
    
    # Set tick marks based on actual data points available
    plt.xticks(range(1, completed_epochs + 1)) 
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(fontsize=12)

    plt.show()


Error: You haven't run the training loop yet! Please run your training loop cell first so there is data to plot.


In [70]:
import torch
model_save_path = "body_mesh_resnet34.pth"

torch.save(model.state_dict(), model_save_path)

print(f"Model successfully saved to {model_save_path}!")


Model successfully saved to body_mesh_resnet34.pth!


In [ ]:
# code to load the model

class PosePredictionModel(nn.Module):
    def __init__(self, output_dim=72):
        super(PosePredictionModel, self).__init__()
        self.backbone = models.resnet34(weights=None) # No need to download ImageNet weights since we are overwriting them
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, output_dim)
        )

    def forward(self, x):
        return self.backbone(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_model = PosePredictionModel(output_dim=72).to(device)

model_save_path = "body_mesh_resnet34.pth"
loaded_model.load_state_dict(torch.load(model_save_path, map_location=device))

loaded_model.eval()

print("Model weights loaded perfectly! Ready for predictions.")


# For continue training

In [ ]:
import torch

checkpoint_path = "body_mesh_checkpoint.pth"

checkpoint = {
    'epoch': epoch + 1,                         # The current epoch number (e.g., 10)
    'model_state_dict': model.state_dict(),       # The trained neural network weights
    'optimizer_state_dict': optimizer.state_dict(), # The optimizer learning rates & momentum
    'train_losses': train_losses,                 # Historic training curves for continuity
    'val_losses': val_losses                      # Historic validation curves for continuity
}

torch.save(checkpoint, checkpoint_path)
print(f"Full training checkpoint successfully saved to {checkpoint_path}!")


In [ ]:
import torch

checkpoint_path = "body_mesh_checkpoint.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = PosePredictionModel(output_dim=72).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

checkpoint = torch.load(checkpoint_path, map_location=device)

model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
start_epoch = checkpoint['epoch']
train_losses = checkpoint['train_losses']
val_losses = checkpoint['val_losses']

print(f"Checkpoint loaded perfectly! Resuming training starting from Epoch {start_epoch + 1}.")
